<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/STA1403_Chapter_12A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_12A:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY _K_ IND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Clustering**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* **12.1 : Introduction**
* **12.2 : _K_ -Means Clustering**
* 12.3 : Hierarchical Clustering
* 12.4 : Advanced

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 26
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Set

Run the next cell to load the data set for this lesson.

In [ ]:
# @title Load Data Set
import pandas as pd

# Load dataset
df_protein = pd.read_csv("https://biologicslab.co/STA1403/data/Protein.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])

print("Data set loaded sucessfully. ✅")

# **Chapter 12 : Clustering**

## **12.1 Introduction**

Linear regression models discussed previously are used to predict the unknown values of the response variable. In these models, the response variable has a central role; the model building process is guided by explaining the variation of the response variable or predicting its values. Therefore, building regression models is known as **supervised learning**. In contrast, building statistical models to identify the underlying structure of data (without focusing on a specific variable) is known as **unsupervised learning**. An important class of unsupervised learning is **clustering**, which is commonly used to identify subgroups within a population. In general, cluster analysis refers to the methods that attempt to divide the data into subgroups such that the observations within the same group are more similar compared to the observations in different groups.

For example, suppose that we believe that while European countries are different with respect to their protein consumption, they could be divided into several groups such that countries within the same group can be considered similar to each other in terms protein consumption. Here, we use the `Protein` data set we discussed earlier. Recall that this data set includes numerical measurements of the protein consumption from 9 different sources: `RedMeat`, `WhiteMeat`, `eggs`, `Milk`, `Fish`, `Cereals`, `Starch` (starchy foods), `nuts` (pulses, nuts, and oil-seeds), and `Fr. Veg` (fruits and vegetables). To start, suppose that we want to group countries according to their consumption of red meat (redMeat) and fish (Fish).

### `Protein` dataset

The `Protein` dataset has been stored in a DataFrame called `df_protein`. Run the next cell to see the first 5 records in this DataFrame.

In [ ]:
# @title Protein dataset

df_protein.head()

The output above shows the first 5 records in the `Protein` data set using Python's `df.head()` method. The dataset contains observations on the consumption of 9 different food groups from 25 European countries.

The core concept in any cluster analysis is the notion of similarity and dissimilarity. It is common to quantify the degree of dissimilarity based on a **distance** measure, which is usually defined for a pair of observations.

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12A_image01A.png)

>**Table** 12.1 Red meat and fish consumption in Albania and Austria

-----------------

>The most commonly used distance measure is the squared distance,
$$ d _ {i j} = \left(x _ {i} - x _ {j}\right) ^ {2}, $$
where $d_{ij}$ refers to the distance between observations $i$ and $j$, $x_{i}$ is the value of random variable $X$ for observation $i$, and $x_{j}$ is the value for observation $j$.

--------------

In the `Protein` data set (Fig. 12.1), the first two countries are Albania and Austria. Suppose we want to measure their degree of dissimilarity (i.e., their distance) in terms of their consumption of red meat and fish (see Table 12.1). The squared distance between these two countries is $ (10.1 - 8.9)^{2} = 1.44 $ in terms of red meat consumption and is $ (0.2 - 2.1)^{2} = 3.61 $ in terms of fish consumption.

To find the overall distance between the two countries, we add the distances based on different variables:
$$ d = 1. 4 4 + 3. 6 1 = 5. 0 5. $$

-----------------

>In general, if we measure $p$ random variables $X_{1},\ldots,X_{p}$, the squared distance between two observations $i$ and $j$ in our sample is
$$
d_{ij} = \left(x_{i1}-x_{j1}\right)^{2} + \dots + \left(x_{ip}-x_{jp}\right)^{2}.
$$
This measure of dissimilarity is called the **squared Euclidean distance**.

----------

## **12.2 $K$-means Clustering**

**$K$-means clustering** is a simple algorithm that uses the squared Euclidean distance as its measure of dissimilarity. We start by specifying the number of clusters (groups) $K$. This is the number of groups we believe exist in the population. Our goal is then to group the $n$ observations in our sample into $K$ clusters such that the overall measure of dissimilarity is small within groups and large between groups. Initially, we divide the observations into $K$ groups randomly. Then the algorithm iteratively improves the clusters.

Let us define the center or centroid of each cluster as an imaginary observation whose measurements are the sample average of all observations in that cluster. For the food consumption example, suppose that the first cluster includes Albania and Austria only. The center of this cluster, denoted as $Center_{1}$, can be regarded as a fictitious country, whose red meat consumption is $9.50$ (average of $10.1$ and $8.9$) and whose fish consumption is $1.15$ (average of $0.2$ and $2.1$).

After randomly partitioning the observations into $K$ groups and finding the center of each cluster, the $K$-means algorithm finds the best clusters by iteratively repeating these steps [11]:
$$
\begin{array}{l}
\text{1. For each observation, find its squared Euclidean distance to all }  \\  
\hspace{1.2em}\text{the } K\text{-centers, and assign it to the cluster with the smallest distance.} \\
\text{2. After regrouping all the observations into } K\text{-clusters, recalculate the } K\text{-centers.} \\
\end{array}
$$



These steps are applied until the clusters do not change (i.e., the centers remain the same after each iteration).

Suppose that we want to cluster the countries into $K = 3$ groups based on their consumption of red meat and fish. Example 1 shows how to do this using Python.

## Example 1: K-Means Cluster Analysis

The code in the cell below uses Python to perform a $K$-Means Cluster Analysis on the consumption of `Fish` and `RedMeat` in the `Protein` dataset.  

In [ ]:
# @title Example 1: K-Means Cluster Analysis

import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = df_protein     # DataFrame
feature_A     = "Fish"
feature_B     = "RedMeat"
n_clusters    = 3              # Specify the number of Groups (K)
# ──────────────────────────────────────────────────────────────────────────────

# Select features
features = [feature_A, feature_B]
X = df[features].values

# Apply K-Means clustering
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(X)

# Get results
labels = kmeans.labels_
centers = kmeans.cluster_centers_

# Add cluster labels to the DataFrame copy
df['ClusterId'] = labels

# Print Results to Console
print(f"\nNumber of clusters: {n_clusters}")
print("\nCluster Centroids:")
print(f"    {feature_A}        {feature_B} ")
print(centers)

print("\nCluster Sizes:")
print(df['ClusterId'].value_counts().sort_index())


If the code is correct, you should see the following output:
```text
Number of clusters: 3

Cluster Centroids:
    Fish        RedMeat
[[ 8.175       8.9125    ]
 [ 1.75454545  7.91818182]
 [ 3.73333333 14.55      ]]

Cluster Sizes:
ClusterId
0     8
1    11
2     6
Name: count, dtype: int64

```

Python created a new variable `ClusterId` with the assignment of each country: 0, 1 or 2. The number of countries assigned to the clusters is shown in the output. The sizes of Cluster 0, Cluster 1, and Cluster 2 are 8, 11, and 6, respectively. There is also a table showing the centroids of these cluster.

## **Exercise 1: K-Means Cluster Analysis**

In the cell below, write the code to perform a $K$-Means Cluster Analysis on the consumption of `Eggs` and `Cereals`.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below.
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A     = "Eggs"
feature_B     = "Cereals"
n_clusters    = 3              # Specify the number of Groups (K)
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 1: K-Means Cluster Analysis




If the code is correct, you should see the following output:
```text
Number of clusters: 3

Cluster Centroids:
    Eggs        Cereals
[[ 2.38571429 39.27142857]
 [ 3.49333333 24.60666667]
 [ 1.43333333 54.06666667]]

Cluster Sizes:
ClusterId
0     7
1    15
2     3
Name: count, dtype: int64

```

We can use a scatterplot to visualize the results of K-means as shown in Example 2.

## Example 2: Plot $K$-Means Clusters

The code in the cell below generates a scatterplot of the $K$-means analysis peformed in  Example 1.

In [ ]:
# @title Example 2: Plot K-Means Clusters

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A     = "Fish"
feature_B     = "RedMeat"
n_clusters    = 3              # Specify the number of Groups (K)
# ──────────────────────────────────────────────────────────────────────────────

# Select features
features = [feature_A, feature_B]
X = df_protein[features].values

# Apply K-Means clustering
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(X)

# Get cluster labels
labels = kmeans.labels_

# Visualize the results
plt.figure(figsize=(7, 6))

# Define the mapping: Cluster ID -> Marker Style
marker_map = {
    0: 'o',   # Circle
    1: '^',   # Triangle
    2: '+' }   # Cross

# Colors for each cluster (using Set1 palette)
colors = ['blue', 'orange', 'green']

for cluster_id in range(n_clusters):
    # Filter points belonging to this cluster
    mask = labels == cluster_id
    x_cluster = X[mask, 0]
    y_cluster = X[mask, 1]

    # Plot the points for this cluster
    plt.scatter(
        x_cluster,
        y_cluster,
        c=colors[cluster_id],
        marker=marker_map[cluster_id],
        s=150,
        alpha=0.7,
        label=f'Cluster {cluster_id}'
    )

# Plot the centroids
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    c='black',
    marker='X',
    s=240,
    edgecolors='white',
    linewidths=2,
    label='Centroids'
)

# Print figure
plt.xlabel(f"{feature_A}", fontsize=14)
plt.ylabel(f"{feature_B}", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12A_image01B.png)

The output shows three clusters represented by blue circles, orange triangles, and green crosses. A black **$X$** is used to mark the center ("centroid") of each cluster. The clusters clearly partition the countries into different groups based on the consumption of fish and red meat. The first cluster (`Cluster 0`) is represented by circles and contains countries whose consumption of both fish and red meat is relatively low. The second cluster (`Cluster 1`) is represented by triangles and includes countries whose consumption of both fish and red meat is relatively high compared to the first group. Finally, the third cluster (`Cluster 2`) is represented by crosses and includes countries whose consumption of red meat tends to be higher compared to the other two groups.

## **Exercise 2: Plot $K$-Means Cluseter**

In the cell below, write the code to visualize these results of the $K$-means analysis of the consumption of `Eggs` and `Cereals` performed in **Exercise 1**.

1. Copy-and-paste Example 2 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
feature_A     = "Eggs"
feature_B     = "Cereals"
n_clusters    = 3              # Specify the number of Groups (K)
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 2: Plot K-Means Clusters





If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_12/chapter_12A_image02B.png)

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()